In [1]:
import numpy as np
import pandas as pd
from imblearn.over_sampling import SMOTE
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    FunctionTransformer,
    OneHotEncoder,
    OrdinalEncoder,
    RobustScaler,
)

# Data Pre-Processing and Feature Engineering
## Goal: Clean, scale, transform, and generate raw features into model-ready arrays.

In [ ]:
class Pre_Processor:

    def __init__(self, model_type="linear", use_smote=True):
        self.model_type = model_type
        self.use_smote = use_smote
        self._label_data()
        self._build_pipelines()

    def _label_data(self):
        # Categorize feature types for specialized preprocessing
        self.numerical_cols = ["age", "avg_glucose_level", "bmi"]
        self.nominal_cols = ["gender", "work_type", "smoking_status"]
        self.binary_cols = [
            "hypertension",
            "heart_disease",
            "ever_married",
            "Residence_type",
        ]
        self.ordinal_cols = []

        # Combine all categorical features for imputing step
        self.all_cat_cols = self.nominal_cols + self.binary_cols

    def _build_pipelines(self):
        # Define base numerical pipeline with median imputer
        steps = [("imputer", SimpleImputer(strategy="median"))]
        
        # Add log transformation and scaling if model is linear/distance-based
        if self.model_type == "linear":
            steps.extend(
                [
                    (
                        "log_transform",
                        FunctionTransformer(np.log1p, validate=False),
                    ),
                    ("scaler", RobustScaler()),
                ]
            )
        elif self.model_type == "tree":
            steps.append(("scaler", RobustScaler()))
            
        self.numerical_pipeline = Pipeline(steps=steps)

        # Categorical imputation and encoding transformers
        self.cat_imputer = SimpleImputer(strategy="most_frequent")
        self.nominal_encoder = OneHotEncoder(
            handle_unknown="ignore", sparse_output=False
        )
        self.binary_encoder = OrdinalEncoder()

    def _remove_duplicates(self, df):
        # Remove identical duplicate rows
        return df.drop_duplicates().reset_index(drop=True)

    def _domain_clean(self, df):
        # Enforce domain logic rules (valid age, non-negative BMI, valid gender)
        valid_age = (df["age"] >= 0) & (df["age"] <= 120)
        valid_bmi = (df["bmi"] > 0) | df["bmi"].isna()
        valid_gender = df["gender"].isin(["Male", "Female"])

        clean_mask = valid_age & valid_bmi & valid_gender
        return df[clean_mask].reset_index(drop=True)

    def _clean_categorical_strings(self, df):
        # Standardize text formatting and convert unknown string labels to NaN
        df_clean = df.copy()
        for col in self.all_cat_cols:
            if col in df_clean.columns:
                df_clean[col] = (
                    df_clean[col]
                    .astype(str)
                    .str.strip()
                    .replace(
                        {
                            "NAN": np.nan,
                            "NaN": np.nan,
                            "Unknown": np.nan,
                            "": np.nan,
                        }
                    )
                )
        return df_clean

    def fit(self, X_train, y_train=None):
        # If target (y_train) is provided, concatenate X and y together to drop bad rows simultaneously
        if y_train is not None:
            train_split = pd.concat([X_train, y_train], axis=1)
            train_split = self._remove_duplicates(train_split)
            train_split = self._domain_clean(train_split)       
            X_clean = train_split.drop(columns=[y_train.name])
        else:
            X_clean = X_train.copy()

        # Clean strings on the training set
        X_clean = self._clean_categorical_strings(X_clean)

        # Fit numerical transformer pipeline on cleaned training data
        self.numerical_pipeline.fit(X_clean[self.numerical_cols])

        # Impute missing categorical values on training data
        cat_imputed_arr = self.cat_imputer.fit_transform(X_clean[self.all_cat_cols])
        cat_imputed_df = pd.DataFrame(cat_imputed_arr, columns=self.all_cat_cols, index=X_clean.index)

        # Fit encoders on imputed categorical training features
        self.nominal_encoder.fit(cat_imputed_df[self.nominal_cols])
        self.binary_encoder.fit(cat_imputed_df[self.binary_cols])
        return self

    def transform(self, X, y=None):
        # NO row dropping here! Keep X (and optional y) completely aligned
        X_clean = X.copy()

        # Clean text strings (value modification only, no rows dropped)
        X_clean = self._clean_categorical_strings(X_clean)

        # Apply fitted numerical transformations (impute/scale/log)
        numerical_processed = self.numerical_pipeline.transform(X_clean[self.numerical_cols])

        # Impute categorical features
        cat_imputed_arr = self.cat_imputer.transform(X_clean[self.all_cat_cols])
        cat_imputed_df = pd.DataFrame(cat_imputed_arr, columns=self.all_cat_cols, index=X_clean.index)

        # Apply fitted encoders
        nominal_processed = self.nominal_encoder.transform(cat_imputed_df[self.nominal_cols])
        binary_processed = self.binary_encoder.transform(cat_imputed_df[self.binary_cols])

        # Stack processed numerical, nominal, and binary columns together
        X_processed_arr = np.hstack([numerical_processed, nominal_processed, binary_processed])

        # Reconstruct output column names
        nominal_feature_names = list(self.nominal_encoder.get_feature_names_out(self.nominal_cols))
        feature_names = self.numerical_cols + nominal_feature_names + self.binary_cols

        # Format output back into a pandas DataFrame
        X_processed = pd.DataFrame(X_processed_arr, columns=feature_names)

        # Return features (and reset y index if y was provided)
        if y is not None:
            return X_processed, y.reset_index(drop=True)
        return X_processed

    def fit_transform(self, X_train, y_train=None):
        # Fit parameters and transform the training split
        X_processed, y_clean = self.fit(X_train, y_train).transform(X_train, y_train)

        # Apply SMOTE class imbalance correction ONLY on the training set
        if y_clean is not None and self.use_smote:
            smote = SMOTE(random_state=42)
            X_resampled_arr, y_resampled = smote.fit_resample(X_processed, y_clean)
            X_resampled = pd.DataFrame(X_resampled_arr, columns=X_processed.columns)
            return X_resampled, y_resampled

        return X_processed, y_clean